In [0]:
# Import everything we need
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, current_timestamp,
    when, lit
)
from pyspark.sql.types import (
    StructType, StructField, StringType, 
    IntegerType, DoubleType, TimestampType
)

# Define the schema for incoming order events
event_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("total_price", DoubleType(), True),
    StructField("order_timestamp", StringType(), True),
    StructField("status", StringType(), True)
])

print("✅ Schema defined — ready to stream!")
print(f"   Fields: {[f.name for f in event_schema.fields]}")

✅ Schema defined — ready to stream!
   Fields: ['event_id', 'customer_id', 'product_id', 'quantity', 'unit_price', 'total_price', 'order_timestamp', 'status']


In [0]:
#Cell 2 — Create a streaming event generator
import random
import uuid
import builtins
from datetime import datetime, timezone
from pyspark.sql import Row

# Explicitly use Python's built-in round, not PySpark's
_round = builtins.round

# Seed product prices
random.seed(42)
PRODUCT_PRICES = {i: _round(random.uniform(10, 500), 2) for i in range(1, 51)}
random.seed()

def generate_events(n=100):
    events = []
    for _ in range(n):
        product_id = random.randint(1, 50)
        unit_price = PRODUCT_PRICES[product_id]
        quantity = random.randint(1, 10)
        total_price = _round(unit_price * quantity, 2)
        status = "returned" if random.random() < 0.10 else "completed"
        events.append(Row(
            event_id=str(uuid.uuid4()),
            customer_id=random.randint(1, 200),
            product_id=product_id,
            quantity=quantity,
            unit_price=float(unit_price),
            total_price=float(total_price),
            order_timestamp=datetime.now(timezone.utc).isoformat(),
            status=status
        ))
    return events

# Test it
sample = generate_events(5)
for e in sample:
    print(f"  customer={e.customer_id:3d}  product={e.product_id:2d}  "
          f"qty={e.quantity}  total=${e.total_price:8.2f}  status={e.status}")

  customer= 17  product=43  qty=4  total=$  486.68  status=completed
  customer= 49  product=18  qty=9  total=$ 2688.66  status=returned
  customer=150  product=16  qty=3  total=$  831.06  status=completed
  customer=101  product=20  qty=4  total=$   52.72  status=completed
  customer= 60  product= 6  qty=6  total=$ 2049.48  status=completed


In [0]:
# Cell 3 — Write events to a Delta table

from pyspark.sql.functions import to_timestamp

# Generate a batch of 100 events
events = generate_events(100)

# Convert to Spark DataFrame
events_df = spark.createDataFrame(events, schema=event_schema)

# Add processing timestamp and cast order_timestamp to proper type
events_df = events_df.withColumn(
    "order_timestamp", to_timestamp(col("order_timestamp"))
).withColumn(
    "processed_at", current_timestamp()
)

# Write to Delta table
events_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.ecommerce.bronze_orders_stream")

print(f"✅ Batch written to bronze_orders_stream")
print(f"   Events: {events_df.count()}")

# Show sample
display(events_df.limit(5))

✅ Batch written to bronze_orders_stream
   Events: 100


event_id,customer_id,product_id,quantity,unit_price,total_price,order_timestamp,status,processed_at
a732cd8a-5be7-4160-97e8-3a724b36ce33,84,17,9,118.02,1062.18,2026-04-14T10:44:27.555Z,completed,2026-04-14T10:44:30.110Z
6659f54c-7e81-4ef1-ab80-ff663a9797b8,33,6,1,341.58,341.58,2026-04-14T10:44:27.555Z,returned,2026-04-14T10:44:30.110Z
2981666a-1e01-4903-be02-22233bc10df0,7,13,7,23.0,161.0,2026-04-14T10:44:27.555Z,completed,2026-04-14T10:44:30.110Z
123fda90-8991-4d07-8d42-4e4c50b8c11e,124,27,2,55.45,110.9,2026-04-14T10:44:27.555Z,completed,2026-04-14T10:44:30.110Z
0ea3b55b-555c-4bd7-8530-d7e558b5cd10,50,9,5,216.74,1083.7,2026-04-14T10:44:27.555Z,returned,2026-04-14T10:44:30.110Z


In [0]:
#Cell 4 Simulate continuous streaming with micro-batches
import time

# Simulate 5 waves of incoming orders
BATCHES = 5
EVENTS_PER_BATCH = 50
SLEEP_SECONDS = 5

print("🚀 Starting streaming simulation...")
print(f"   {BATCHES} batches × {EVENTS_PER_BATCH} events = {BATCHES * EVENTS_PER_BATCH} total events")
print("-" * 50)

for batch_num in range(1, BATCHES + 1):
    # Generate fresh events
    events = generate_events(EVENTS_PER_BATCH)
    df = spark.createDataFrame(events, schema=event_schema)
    df = df.withColumn(
        "order_timestamp", to_timestamp(col("order_timestamp"))
    ).withColumn(
        "processed_at", current_timestamp()
    )
    
    # Append to streaming Delta table
    df.write.format("delta").mode("append") \
        .saveAsTable("workspace.ecommerce.bronze_orders_stream")
    
    # Count total so far
    total = spark.table("workspace.ecommerce.bronze_orders_stream").count()
    
    print(f"  Batch {batch_num}/{BATCHES} → {EVENTS_PER_BATCH} events written "
          f"| Total in table: {total}")
    
    if batch_num < BATCHES:
        time.sleep(SLEEP_SECONDS)

print("-" * 50)
print("✅ Streaming simulation complete!")

🚀 Starting streaming simulation...
   5 batches × 50 events = 250 total events
--------------------------------------------------
  Batch 1/5 → 50 events written | Total in table: 150
  Batch 2/5 → 50 events written | Total in table: 200
  Batch 3/5 → 50 events written | Total in table: 250
  Batch 4/5 → 50 events written | Total in table: 300
  Batch 5/5 → 50 events written | Total in table: 350
--------------------------------------------------
✅ Streaming simulation complete!


In [0]:
# Cell 5 — Query the streaming table live

from pyspark.sql.functions import count, sum, avg, window

# Current state of the streaming table
stream_df = spark.table("workspace.ecommerce.bronze_orders_stream")

print(f"📊 Total events in stream: {stream_df.count()}")
print()

# Revenue by status
print("=== Revenue by Status ===")
display(
    stream_df.groupBy("status")
    .agg(
        count("*").alias("order_count"),
        sum("total_price").alias("total_revenue"),
        avg("total_price").alias("avg_order_value")
    )
    .orderBy("total_revenue", ascending=False)
)

📊 Total events in stream: 350

=== Revenue by Status ===


status,order_count,total_revenue,avg_order_value
completed,318,431444.31,1356.7431132075471
returned,32,43953.43,1373.5446875


In [0]:
# Cell 6 — Time-windowed aggregations
from pyspark.sql.functions import window, date_trunc

# Revenue by minute — shows event flow over time
print("=== Events by Processing Minute ===")
display(
    stream_df
    .groupBy(date_trunc("minute", col("processed_at")).alias("minute"))
    .agg(
        count("*").alias("events"),
        sum("total_price").alias("revenue")
    )
    .orderBy("minute")
)

=== Events by Processing Minute ===


minute,events,revenue
2026-04-14T10:10:00.000Z,100,132741.02999999997
2026-04-14T10:16:00.000Z,250,342656.70999999996


In [0]:
# Cell 7 — Anomaly detection on the stream

# from pyspark.sql.functions import stddev, mean

# Calculate baseline statistics
stats = stream_df.agg(
    mean("total_price").alias("mean_price"),
    stddev("total_price").alias("stddev_price")
).collect()[0]

mean_price = stats["mean_price"]
stddev_price = stats["stddev_price"]
threshold = mean_price + (2 * stddev_price)

print(f"📊 Baseline Statistics:")
print(f"   Mean order value:   ${mean_price:,.2f}")
print(f"   Std deviation:      ${stddev_price:,.2f}")
print(f"   Anomaly threshold:  ${threshold:,.2f} (mean + 2σ)")
print()

# Find anomalous orders
anomalies = stream_df.filter(col("total_price") > threshold)

print(f"🚨 Anomalous orders detected: {anomalies.count()}")
print()
display(
    anomalies.select(
        "event_id", "customer_id", "product_id",
        "quantity", "total_price", "status", "processed_at"
    ).orderBy("total_price", ascending=False)
)

📊 Baseline Statistics:
   Mean order value:   $1,358.28
   Std deviation:      $1,161.28
   Anomaly threshold:  $3,680.83 (mean + 2σ)

🚨 Anomalous orders detected: 15



event_id,customer_id,product_id,quantity,total_price,status,processed_at
2f99d310-7ebb-4a74-8798-6e577caad775,160,34,10,4868.3,completed,2026-04-14T10:16:30.135Z
524c133c-87da-4c96-934e-526dde285534,172,25,10,4790.3,completed,2026-04-14T10:10:45.770Z
f062712c-aecd-474b-96ac-ed447df42b18,94,25,10,4790.3,completed,2026-04-14T10:16:21.898Z
284c1d4b-74ff-4314-93ba-5b04e57fcaf5,76,34,9,4381.47,returned,2026-04-14T10:16:21.898Z
7d51bff6-72b5-40c4-8a82-9a6416910c74,20,39,10,4322.4,completed,2026-04-14T10:16:21.898Z
d2e0839b-745f-4f4a-a442-57797d510f83,71,39,10,4322.4,completed,2026-04-14T10:10:45.770Z
07edc27e-5bfd-42b8-86ba-d72ce1672620,8,29,10,4252.7,completed,2026-04-14T10:16:52.570Z
efcf7fd2-34cf-4483-9f6d-d2a17a704f34,96,37,10,4164.1,returned,2026-04-14T10:10:45.770Z
ac77b5ec-335f-4907-abe1-99f829f5c1b8,128,31,10,4054.9,completed,2026-04-14T10:10:45.770Z
26762518-e0c3-403e-86dd-4dbaa5e08434,94,21,10,4048.5,completed,2026-04-14T10:16:45.034Z


In [0]:
# Save anomalies to their own Delta table
anomalies.withColumn(
    "anomaly_threshold", lit(threshold)
).withColumn(
    "deviation", col("total_price") - lit(mean_price)
).write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.ecommerce.gold_stream_anomalies")

print(f"✅ {anomalies.count()} anomalies saved to gold_stream_anomalies")
print(f"   Mean order value:  ${mean_price:,.2f}")
print(f"   Threshold applied: ${threshold:,.2f}")
print()

display(
    spark.table("workspace.ecommerce.gold_stream_anomalies")
    .select("customer_id", "product_id", "quantity", 
            "total_price", "deviation", "status")
    .orderBy("deviation", ascending=False)
)

✅ 15 anomalies saved to gold_stream_anomalies
   Mean order value:  $1,358.28
   Threshold applied: $3,680.83



customer_id,product_id,quantity,total_price,deviation,status
160,34,10,4868.3,3510.020742857143,completed
94,25,10,4790.3,3432.020742857143,completed
172,25,10,4790.3,3432.020742857143,completed
76,34,9,4381.47,3023.190742857143,returned
20,39,10,4322.4,2964.1207428571424,completed
71,39,10,4322.4,2964.1207428571424,completed
8,29,10,4252.7,2894.4207428571426,completed
96,37,10,4164.1,2805.820742857143,returned
128,31,10,4054.9,2696.6207428571433,completed
94,21,10,4048.5,2690.220742857143,completed


In [0]:
# Cell 9 — Build a streaming summary dashboard query

# Executive summary of the stream
total_events = stream_df.count()
total_revenue = stream_df.agg({"total_price": "sum"}).collect()[0][0]
total_returns = stream_df.filter(col("status") == "returned").count()
total_anomalies = anomalies.count()
avg_order = stream_df.agg({"total_price": "avg"}).collect()[0][0]

print("=" * 50)
print("   STREAMING PIPELINE — EXECUTIVE SUMMARY")
print("=" * 50)
print(f"  Total events processed:  {total_events:,}")
print(f"  Total revenue:           ${total_revenue:,.2f}")
print(f"  Average order value:     ${avg_order:,.2f}")
print(f"  Returns:                 {total_returns} ({total_returns/total_events*100:.1f}%)")
print(f"  Anomalies detected:      {total_anomalies} ({total_anomalies/total_events*100:.1f}%)")
print(f"  Revenue at risk:         ${anomalies.agg({'total_price':'sum'}).collect()[0][0]:,.2f}")
print("=" * 50)

   STREAMING PIPELINE — EXECUTIVE SUMMARY
  Total events processed:  350
  Total revenue:           $475,397.74
  Average order value:     $1,358.28
  Returns:                 32 (9.1%)
  Anomalies detected:      15 (4.3%)
  Revenue at risk:         $63,754.75
